<a href="https://colab.research.google.com/github/vuhamodala/Float-Chat-for-Indian-ocean-Data/blob/main/XAI_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df23 = pd.read_csv("argo_2023.csv")
df24 = pd.read_csv("argo_2024.csv")
df25 = pd.read_csv("argo_2025.csv")

/tmp/ipykernel_1491/3687920926.py:3: DtypeWarning: Columns (0,1,4,5,6,7,8,9,10,11,12,13,14,15,16,17,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df23 = pd.read_csv("argo_2023.csv")


In [2]:
sample_size = 48000

df23 = df23.sample(n=sample_size, random_state=42)
df24 = df24.sample(n=sample_size, random_state=42)
df25 = df25.sample(n=sample_size, random_state=42)

df = pd.concat([df23, df24, df25], ignore_index=True)

In [3]:
df.columns = df.columns.str.strip().str.lower()

In [4]:
df = df.rename(columns={
    "temp": "temperature",
    "psal": "salinity",
    "pres": "pressure",
    "time": "timestamp",
    "platform_number": "platform_id"
})

In [5]:
df = df[
    (df["temp_qc"] == 1) &
    (df["psal_qc"] == 1) &
    (df["pres_qc"] == 1)
]

In [6]:
df = df.drop(columns=[
    "temp_qc", "psal_qc", "pres_qc",
    "position_qc", "time_qc",
])

In [7]:
df = df.dropna(subset=["temperature", "salinity", "pressure"])

In [8]:
df["temperature"] = pd.to_numeric(df["temperature"], errors="coerce")
df["salinity"] = pd.to_numeric(df["salinity"], errors="coerce")
df["pressure"] = pd.to_numeric(df["pressure"], errors="coerce")

In [10]:
df = df[(df["temperature"] > -2) & (df["temperature"] < 40)]
df = df[(df["salinity"] > 0) & (df["salinity"] < 50)]
df = df[df["pressure"] >= 0]

In [13]:
df

,n_points,cycle_number,data_mode,direction,platform_id,pressure,pres_error,salinity,psal_error,temperature,temp_error,latitude,longitude,timestamp,region,year,month,hour
0,4400,26,D,D,1902267,490.30,2.001,35.12250,0.004,12.401,0.001,-31.10679,50.61055,2023-09-04 16:38:58,Southern IO,2023,9,16
1,17573,18,D,D,1902268,240.30,2.001,35.45400,0.004,15.146,0.001,-25.15532,57.43595,2023-07-21 09:49:31,Southern IO,2023,7,9
2,11022,209,D,A,2901862,392.12,2.4,34.92252,0.01,10.321,0.002,-3.759,89.605,2023-06-23 00:25:34,Equatorial IO,2023,6,0
3,26349,357,R,A,1901701,352.00,NaN,35.24700,NaN,13.192,NaN,-33.21171,56.39564,2023-07-31 22:20:42,Southern IO,2023,7,22
4,19716,33,D,A,1902267,244.60,2.001,35.43920,0.004,14.929,0.001,-31.38812,50.22469,2023-11-22 20:29:09,Southern IO,2023,11,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143995,18656,79,A,A,1902455,108.22,NaN,35.27200,NaN,21.881,NaN,2.1972,77.1318,2025-07-23 16:10:07,Equatorial IO,2025,7,16
143996,10708,77,R,A,4903633,73.60,NaN,35.66300,NaN,19.578,NaN,-31.790068,61.306945,2025-03-15 18:28:30,Southern IO,2025,3,18
143997,1338,79,R,A,7901125,155.10,NaN,34.82700,NaN,17.219,NaN,13.25,90.566667,2025-11-08 14:13:22,Bay of Bengal,2025,11,14
143998,18647,133,A,D,1902475,367.90,NaN,35.33731,NaN,13.789,NaN,-30.69493,56.11607,2025-01-24 13:09:48,Southern IO,2025,1,13


In [14]:
df.isnull().sum()

,0
n_points,0
cycle_number,0
data_mode,0
direction,0
platform_id,0
pressure,0
pres_error,64372
salinity,0
psal_error,64372
temperature,0


In [15]:
df = df.drop(columns=[
    "temp_error",
    "psal_error",
    "pres_error"
])


In [16]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Duplicates removed:", before - after)

Duplicates removed: 1044


In [19]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["date"] = df["timestamp"].dt.date

In [20]:
df

,n_points,cycle_number,data_mode,direction,platform_id,pressure,salinity,temperature,latitude,longitude,timestamp,region,year,month,hour,date
0,4400,26,D,D,1902267,490.30,35.12250,12.401,-31.10679,50.61055,2023-09-04 16:38:58,Southern IO,2023,9,16,2023-09-04
1,17573,18,D,D,1902268,240.30,35.45400,15.146,-25.15532,57.43595,2023-07-21 09:49:31,Southern IO,2023,7,9,2023-07-21
2,11022,209,D,A,2901862,392.12,34.92252,10.321,-3.759,89.605,2023-06-23 00:25:34,Equatorial IO,2023,6,0,2023-06-23
3,26349,357,R,A,1901701,352.00,35.24700,13.192,-33.21171,56.39564,2023-07-31 22:20:42,Southern IO,2023,7,22,2023-07-31
4,19716,33,D,A,1902267,244.60,35.43920,14.929,-31.38812,50.22469,2023-11-22 20:29:09,Southern IO,2023,11,20,2023-11-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143995,18656,79,A,A,1902455,108.22,35.27200,21.881,2.1972,77.1318,2025-07-23 16:10:07,Equatorial IO,2025,7,16,2025-07-23
143996,10708,77,R,A,4903633,73.60,35.66300,19.578,-31.790068,61.306945,2025-03-15 18:28:30,Southern IO,2025,3,18,2025-03-15
143997,1338,79,R,A,7901125,155.10,34.82700,17.219,13.25,90.566667,2025-11-08 14:13:22,Bay of Bengal,2025,11,14,2025-11-08
143998,18647,133,A,D,1902475,367.90,35.33731,13.789,-30.69493,56.11607,2025-01-24 13:09:48,Southern IO,2025,1,13,2025-01-24


In [21]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce").dt.time

In [22]:
df

,n_points,cycle_number,data_mode,direction,platform_id,pressure,salinity,temperature,latitude,longitude,timestamp,region,year,month,hour,date
0,4400,26,D,D,1902267,490.30,35.12250,12.401,-31.10679,50.61055,16:38:58,Southern IO,2023,9,16,2023-09-04
1,17573,18,D,D,1902268,240.30,35.45400,15.146,-25.15532,57.43595,09:49:31,Southern IO,2023,7,9,2023-07-21
2,11022,209,D,A,2901862,392.12,34.92252,10.321,-3.759,89.605,00:25:34,Equatorial IO,2023,6,0,2023-06-23
3,26349,357,R,A,1901701,352.00,35.24700,13.192,-33.21171,56.39564,22:20:42,Southern IO,2023,7,22,2023-07-31
4,19716,33,D,A,1902267,244.60,35.43920,14.929,-31.38812,50.22469,20:29:09,Southern IO,2023,11,20,2023-11-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143995,18656,79,A,A,1902455,108.22,35.27200,21.881,2.1972,77.1318,16:10:07,Equatorial IO,2025,7,16,2025-07-23
143996,10708,77,R,A,4903633,73.60,35.66300,19.578,-31.790068,61.306945,18:28:30,Southern IO,2025,3,18,2025-03-15
143997,1338,79,R,A,7901125,155.10,34.82700,17.219,13.25,90.566667,14:13:22,Bay of Bengal,2025,11,14,2025-11-08
143998,18647,133,A,D,1902475,367.90,35.33731,13.789,-30.69493,56.11607,13:09:48,Southern IO,2025,1,13,2025-01-24


In [23]:
df = df.rename(columns={"timestamp": "time"})

In [24]:
df = df.drop(columns=["month", "hour"])

In [25]:
contextual_cols = [
    "temperature",
    "salinity",
    "pressure",
    "latitude",
    "longitude",
    "year",
    "time",
    "region"
]

metadata_cols = [
    "platform_id",
    "cycle_number",
    "n_points",
    "data_mode",
    "direction",

]

In [26]:
df["record_id"] = range(len(df))

In [27]:
metadata_cols.insert(0, "record_id")
contextual_cols.insert(0, "record_id")

In [28]:
metadata_df = df[metadata_cols]
contextual_df = df[contextual_cols]

In [29]:
metadata_df.to_csv("metadata.csv", index=False)
contextual_df.to_csv("contextual_data.csv", index=False)